This is the working one

Data source: https://data.humdata.org/dataset/international-migration-flows

- **CL**:
    - December 2020:  
The Chilean Constitutional Court reviewed and approved the final version after challenges to several articles.
    - 11 April 2021:  
President Piñera officially promulgated the law.
    - 20 April 2021:  
The law was published in the Diario Oficial.

In [ ]:
import io
import re
import numpy as np
import pandas as pd
from tqdm import tqdm
import geopandas as gpd
from pathlib import Path
from itertools import permutations

# Visualisation
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

import pandas as pd
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'  # has wide Unicode support

# Base directories
BASE_DIR = Path("/Users/wenlanzhang/PycharmProjects/Mapineq/src/")
DATA_DIR = Path("/Users/wenlanzhang/Downloads/PhD_UCL/Data/Oxford")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from patsy import dmatrices
import statsmodels.api as sm
from patsy import dmatrices
# Required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from patsy import dmatrices

# --- SETTINGS (change if needed) ---
df = BACI_df.copy()   # your dataframe
treated_pair_name = 'CL->US'   # treated pair
time_col = 'time_index'        # integer time index (1..T)
pair_col = 'pair'
y_col = 'log_migrants'
ym_col = 'year_month'        # month FE column used in regressions


In [ ]:
df = pd.read_csv(DATA_DIR/f"Migration/international_migration_flow.csv") 
df['year'] = pd.to_datetime(df['migration_month']).dt.year
df['month'] = pd.to_datetime(df['migration_month']).dt.month
df["migration_month"] = pd.to_datetime(df["migration_month"])

# Unique list from both columns
unique_countries = pd.unique(df[["country_from", "country_to"]].values.ravel())

df

In [ ]:
cl_from = df[df['country_from'] == 'CL']

# Compute total migrants by destination
top10_countries = (
    cl_from.groupby('country_to')['num_migrants']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)
top10_countries

In [ ]:
# top10_countries
df_top10 = cl_from[cl_from['country_to'].isin(top10_countries)]
df_top10

In [ ]:
# set the intervention date 
intervention_date = pd.to_datetime("2021-04-01")

treated = 'US'

CL_top10 = df_top10.copy()

In [ ]:
# create a pair identifier (origin-destination)
CL_top10['pair'] = CL_top10['country_from'].astype(str) + "->" + CL_top10['country_to'].astype(str)
# create a time id for month (string or period)
CL_top10['year_month'] = CL_top10['migration_month'].dt.to_period('M').astype(str)
# also useful numeric time index
CL_top10['time_index'] = (CL_top10['migration_month'].dt.year - CL_top10['migration_month'].dt.year.min()) * 12 + CL_top10['migration_month'].dt.month

# create a pair identifier (origin-destination)
CL_top10['pair'] = CL_top10['country_from'].astype(str) + "->" + CL_top10['country_to'].astype(str)

# create a time id for month (string or period)
CL_top10['year_month'] = CL_top10['migration_month'].dt.to_period('M').astype(str)
# also useful numeric time index
CL_top10['time_index'] = (CL_top10['migration_month'].dt.year - CL_top10['migration_month'].dt.year.min()) * 12 + CL_top10['migration_month'].dt.month

# CL_top10

# treatment dummy: only CL -> US is treated
CL_top10['treated_pair'] = ((CL_top10['country_from'] == 'CL') & (CL_top10['country_to'] == 'US')).astype(int)

# post dummy: after (>=) the intervention month
CL_top10['post'] = (CL_top10['migration_month'] >= intervention_date).astype(int)

# DID interaction
CL_top10['did'] = CL_top10['treated_pair'] * CL_top10['post']

# counts often skewed: log transform (add 1 to avoid log(0))
CL_top10['log_migrants'] = np.log(CL_top10['num_migrants'] + 1)

CL_top10

## Setting 

In [ ]:
event_date = pd.to_datetime("2021-04-01")
treated_pair = 'CL->US'   

## Select countried 

In [ ]:
CL_US_df = df[(df['country_to'] == 'US') | (df['country_from'] == 'CL')]

# get total migrants for each country pair
pair_totals = CL_US_df.groupby(['country_from', 'country_to'])['num_migrants'].sum().reset_index()

cl_us_total = pair_totals.loc[
    (pair_totals['country_from'] == 'CL') & (pair_totals['country_to'] == 'US'),
    'num_migrants'
].iloc[0]

print(f"Total migrants from CL to US: {cl_us_total}")
pair_totals

In [ ]:
# Top 10 most similar "CL -> any country" (exclude US itself)
cl_to_any = pair_totals[pair_totals['country_from'] == 'CL']
cl_to_any = cl_to_any[cl_to_any['country_to'] != 'US']  # exclude US
top10_cl_to_any = cl_to_any.iloc[(cl_to_any['num_migrants'] - cl_us_total).abs().argsort()[:10]]
top10_cl_to_any

In [ ]:
# Top 10 most similar "any country -> US" (exclude CL itself)
any_to_us = pair_totals[pair_totals['country_to'] == 'US']

exclude_countries = ['CL', 'RU', 'UA', 'AF', 'HT'] # Exclude specific origin countries
any_to_us = any_to_us[~any_to_us['country_from'].isin(exclude_countries)]

top10_any_to_us = any_to_us.iloc[(any_to_us['num_migrants'] - cl_us_total).abs().argsort()[:10]]
top10_any_to_us

In [ ]:
# --- Build list of unique (country_from, country_to) pairs from top10s and CL->US
pairs = pd.concat([
    top10_cl_to_any[['country_from', 'country_to']],   # CL → any
    top10_any_to_us[['country_from', 'country_to']],   # any → US
    CL_US_df.query("country_from == 'CL' and country_to == 'US'")[['country_from', 'country_to']]  # CL → US
], ignore_index=True).drop_duplicates().reset_index(drop=True)

# --- Filter CL_US_df to keep only rows for those pairs (month-level)
df_control = CL_US_df.merge(pairs, on=['country_from', 'country_to'], how='inner')
df_control

## Check Plot

In [ ]:
# Create a combined pair label for clarity
df_selected = df_control.copy()
df_selected['pair'] = df_selected['country_from'] + ' → ' + df_selected['country_to']

fig = px.line(
    df_selected,
    x='migration_month',
    y='num_migrants',
    color='pair',  # color by full country pair
    title='Migration Trends Over Time by Country Pair',
    labels={
        'migration_month': 'Month',
        'num_migrants': 'Number of Migrants',
        'pair': 'Country Pair'
    },
)

# Add vertical line (e.g., key policy change date)
fig.add_vline(
    x="2021-04-01",
    line_width=2,
    line_dash="dash",
    line_color="red"
)

# Add annotation
fig.add_annotation(
    x="2021-04-01",
    y=df_selected['num_migrants'].max(),
    text="Ley 21325<br>April 2021",
    showarrow=True,
    arrowhead=2,
    ax=40,
    ay=-40,
    font=dict(color="red", size=12),
    align="center"
)

# Style and layout tweaks
fig.update_layout(
    template='plotly_white',
    legend_title_text='Country Pair',
    xaxis_title='Migration Month',
    yaxis_title='Number of Migrants',
    # hovermode='x unified'
    hovermode='closest'  # only show the hovered line
)

fig.show()

## Setting 

In [ ]:
# --- 2. Create 'pair' and 'year_month' ---
df_selected['pair'] = df_selected['country_from'] + '->' + df_selected['country_to']
df_selected['year_month'] = df_selected['migration_month'].dt.to_period('M').astype(str)

# --- 3. Create time index (1, 2, 3, ...) based on chronological order ---
df_selected = df_selected.sort_values('migration_month')
df_selected['time_index'] = df_selected['migration_month'].rank(method='dense').astype(int)

# --- 5. Add BACI variables ---
df_selected['treated_pair'] = np.where(df_selected['pair'] == treated_pair, 1, 0)
df_selected['post'] = np.where(df_selected['migration_month'] >= event_date, 1, 0)

# Interaction term: DID = treated × post
df_selected['did'] = df_selected['treated_pair'] * df_selected['post']

# --- 6. Log-transform migrants (handle zeros safely) ---
df_selected['log_migrants'] = np.log1p(df_selected['num_migrants'])  # log(1 + x) to avoid log(0)

# --- 7. Optional: reorder columns to match your example ---
df_selected = df_selected[[
    'country_from', 'country_to', 'migration_month', 'num_migrants',
    'year', 'month', 'pair', 'year_month', 'time_index',
    'treated_pair', 'post', 'did', 'log_migrants'
]]

# --- 8. Check the output ---
df_selected

In [ ]:
BACI_df = df_selected.copy()

# DiD regression (OLS on log outcome) with pair and time fixed effects
We'll include:
- did (treatment*post) as coefficient of interest
- pair fixed effects (C(pair))
- time fixed effects (C(year_month))
- and cluster standard errors at the pair level.

In [ ]:
# # DID with raw counts
# # formula_raw = 'num_migrants ~ did + C(pair) + C(year_month)'
# formula_raw = 'num_migrants ~ 0 + did + C(pair) + C(year_month)'

# # Construct design matrices
# y_raw, X_raw = dmatrices(formula_raw, BACI_df, return_type='dataframe')

# # Fit OLS with cluster-robust SE clustered by pair
# model_ols_raw = sm.OLS(y_raw, X_raw)
# res_ols_raw = model_ols_raw.fit(cov_type='cluster', cov_kwds={'groups': BACI_df['pair']})

# # Print coefficient table
# print(res_ols_raw.summary().tables[1])

# # Extract DID estimate
# did_coef_raw = res_ols_raw.params.get('did', None)
# did_se_raw = res_ols_raw.bse.get('did', None)
# print(f"\nDID (raw outcome) coef = {did_coef_raw:.4f}, SE (clustered by pair) = {did_se_raw:.4f}")

# # Interpretation: absolute change in counts
# if did_coef_raw is not None:
#     print(f"Approx change in number of migrants (treated vs controls, after vs before): {did_coef_raw:.2f}")

In [ ]:
# Formula with log
# formula = 'log_migrants ~ did + C(pair) + C(year_month)'
formula = 'log_migrants ~ 0 + did + C(pair) + C(year_month)'

# Use patsy to construct matrices and then fit OLS without intercept (because dummies will absorb)
y, X = dmatrices(formula, BACI_df, return_type='dataframe')

# Fit OLS with cluster-robust SE clustered by pair
model_ols = sm.OLS(y, X)
res_ols = model_ols.fit(cov_type='cluster', cov_kwds={'groups': BACI_df['pair']})
print(res_ols.summary().tables[1])

# Extract DID estimate
did_coef = res_ols.params.get('did', None)
did_se = res_ols.bse.get('did', None)
print(f"\nDID (log outcome) coef = {did_coef:.4f}, SE (clustered by pair) = {did_se:.4f}")

# Interpretation: exp(coef)-1 ~ percent change in counts
if did_coef is not None:
    approx_pct = (np.exp(did_coef)-1)*100
    print(f"Approx % change in counts (treated vs controls, after vs before): {approx_pct:.2f}%")

In [ ]:
# # Raw mean 
# plot_raw = BACI_df.groupby(['year_month', 'treated_pair'])['num_migrants'].mean().reset_index()
# plt.figure(figsize=(10,4))
# sns.lineplot(data=plot_raw, x='year_month', y='num_migrants', hue='treated_pair')
# plt.title('Raw mean migrants: treated vs control')
# plt.axvline(27, color='k', ls='--', label='Event')
# plt.xticks(rotation=90)
# plt.tight_layout()
# plt.show()

In [ ]:
# log_migrants
plot_log = BACI_df.groupby(['year_month', 'treated_pair'])['log_migrants'].mean().reset_index()
plt.figure(figsize=(10,4))
sns.lineplot(data=plot_log, x='year_month', y='log_migrants', hue='treated_pair', estimator='mean')
plt.axvline(27, color='k', ls='--')
plt.title('Log migrants: treated vs control')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

1. Parallel trends — visual + formal
- Why: central DID assumption.
- How: event-study with many pre-period leads; joint F-test on pre-treatment dummies (all leads = 0).
- Code hint:
- Show: event-study plot with 95% CIs; table with F-stat and p-value.
- Wording: “Pre-treatment coefficients are statistically indistinguishable from zero (joint F = X, p = Y), supporting parallel trends.”

# Check result

In [ ]:
# 0. Quick checks
print("Unique pairs:", df[pair_col].nunique())
assert treated_pair_name in df[pair_col].unique(), "Treated pair not in df!"

# get treatment period (assume 'post' becomes 1 in treated pair after treatment)
# If you have a common treatment time index known, replace this
treated_mask = (df[pair_col] == treated_pair_name) & (df['post'] == 1)
if treated_mask.any():
    treatment_time = df.loc[treated_mask, time_col].min()
else:
    # fallback: if you know treatment_time manually, set it here:
    treatment_time = int(df[time_col].max()/2)  # placeholder; change as needed
print("Assumed treatment_time (time_index) =", treatment_time)

In [ ]:
df[df['pair'] == 'CL->US']

## Parallel trends plot (group means) 

In [ ]:
# create group labels: treated vs all_controls vs control subgroups
df['group'] = np.where(df[pair_col] == treated_pair_name, 'Treated (CL->US)', 'Control (all)')
# optional subgroup splits:
df['control_subgroup'] = np.where((df[pair_col].str.startswith('CL->')) & (df[pair_col] != treated_pair_name),
                                  'CL->X', np.where((df[pair_col].str.endswith('->US')) & (df[pair_col] != treated_pair_name),
                                                     'X->US', 'Other'))
# compute means by time
mean_by_time = df.groupby([time_col, 'group'])[y_col].mean().reset_index()
mean_sub = df.groupby([time_col, 'control_subgroup'])[y_col].mean().reset_index()

plt.figure(figsize=(10,5))
# plot controls mean
ctrl = mean_by_time[mean_by_time['group']=='Control (all)']
plt.plot(ctrl[time_col], ctrl[y_col], label='Control (all pairs)', linewidth=2, linestyle='--')
# plot treated
t = mean_by_time[mean_by_time['group']=='Treated (CL->US)']
plt.plot(t[time_col], t[y_col], label='Treated: CL->US', linewidth=3, color='C1')
# optional: subgroups
for name, g in mean_sub.groupby('control_subgroup'):
    if name == 'Other': continue
    plt.plot(g[time_col], g[y_col], label=name, linewidth=1, alpha=0.7)
plt.axvline(treatment_time-0.5, color='k', linestyle=':', label='Treatment')
plt.xlabel('Time index')
plt.ylabel('Average log migrants')
plt.title('Parallel trends: Treated vs Controls')
plt.legend()
plt.tight_layout()
plt.show()

## Spaghetti plot: all pairs, highlight treated

In [ ]:
plt.figure(figsize=(11,6))
pairs = df[pair_col].unique()
for p in pairs:
    series = df[df[pair_col] == p].sort_values(time_col)
    if p == treated_pair_name:
        plt.plot(series[time_col], series[y_col], linewidth=3.0, label=p, color='C1')
    else:
        plt.plot(series[time_col], series[y_col], color='gray', alpha=0.3)
plt.axvline(treatment_time-0.5, color='k', linestyle=':', label='Treatment')
plt.xlabel('Time index')
plt.ylabel('log migrants')
plt.title('Spaghetti: pairs time-series (treated highlighted)')
plt.legend()
plt.tight_layout()
plt.show()

## Placebo (in-space) permutation test — treat each control pair as “treated”

This iteratively treats each control pair as if it were the treated unit at the same treatment time and collects placebo DID estimates. It plots the distribution and computes the empirical p-value.

In [ ]:
# Base DID formula (your original)
base_formula = 'log_migrants ~ did + C({}) + C({}) - 1'.format(pair_col, ym_col)

placebo_results = []
pairs = sorted(df[pair_col].unique())
pairs.remove(treated_pair_name)   # controls only

for p in pairs:
    tmp = df.copy()
    tmp['placebo_treated'] = (tmp[pair_col] == p).astype(int)
    tmp['placebo_post'] = (tmp[time_col] >= treatment_time).astype(int)   # use same treatment_time
    tmp['placebo_did'] = tmp['placebo_treated'] * tmp['placebo_post']
    try:
        y_p, X_p = dmatrices('log_migrants ~ placebo_did + C({}) + C({}) - 1'.format(pair_col, ym_col), tmp, return_type='dataframe')
        mod_p = sm.OLS(y_p, X_p).fit(cov_type='cluster', cov_kwds={'groups': tmp[pair_col]})
        coef_p = mod_p.params.get('placebo_did', np.nan)
        se_p = mod_p.bse.get('placebo_did', np.nan)
        placebo_results.append({'pair': p, 'coef': coef_p, 'se': se_p})
    except Exception as e:
        placebo_results.append({'pair': p, 'coef': np.nan, 'se': np.nan})

placebo_df = pd.DataFrame(placebo_results).dropna()

# Get actual DID from original specification
y0, X0 = dmatrices('log_migrants ~ did + C({}) + C({}) - 1'.format(pair_col, ym_col), df, return_type='dataframe')
orig_mod = sm.OLS(y0, X0).fit(cov_type='cluster', cov_kwds={'groups': df[pair_col]})
actual_coef = orig_mod.params.get('did', np.nan)
print("Actual DID coef:", actual_coef)

# Empirical p-value (two-sided)
emp_p_two = (np.sum(np.abs(placebo_df['coef']) >= np.abs(actual_coef)) + 1) / (len(placebo_df) + 1)
# One-sided (actual in upper tail)
emp_p_one = (np.sum(placebo_df['coef'] >= actual_coef) + 1) / (len(placebo_df) + 1)

print(f"Placebo distribution: N_placebos = {len(placebo_df)}")
print(f"Empirical p-value (two-sided) = {emp_p_two:.3f}")
print(f"Empirical p-value (one-sided upper) = {emp_p_one:.3f}")

# Plot histogram with actual coef marked
plt.figure(figsize=(8,4))
plt.hist(placebo_df['coef'], bins=12, alpha=0.8)
plt.axvline(actual_coef, color='red', linestyle='--', lw=2, label='Actual DID')
plt.xlabel('Placebo DID coefficient (log scale)')
plt.ylabel('Count')
plt.title('In-space placebo distribution (each control pair treated)')
plt.legend()
plt.show()

In [ ]:
# --- 4) Placebo / permutation: estimate DID as if each control pair was treated ---
placebo_coefs = []
pairs_list = [p for p in df[pair_col].unique() if p != treated_pair_name]

# Simple DID formula used in your original model with pair and time FE
formula_base = 'log_migrants ~ did + C(pair) + C(year_month) - 1'

for p in pairs_list:
    tmp = df.copy()
    # create placebo treated_pair: treat pair p as if treated
    tmp['placebo_treated'] = (tmp[pair_col] == p).astype(int)
    # assume same treatment_time: set 'post' according to same treatment_time
    tmp['placebo_post'] = (tmp[time_col] >= treatment_time).astype(int)
    tmp['placebo_did'] = tmp['placebo_treated'] * tmp['placebo_post']
    try:
        y, X = dmatrices('log_migrants ~ placebo_did + C(pair) + C(year_month) - 1', tmp, return_type='dataframe')
        mod = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': tmp[pair_col]})
        coef_p = mod.params.get('placebo_did', np.nan)
        placebo_coefs.append({'pair': p, 'coef': coef_p})
    except Exception as e:
        placebo_coefs.append({'pair': p, 'coef': np.nan})
        # continue

placebo_df = pd.DataFrame(placebo_coefs).dropna()

# get actual DID estimate from original model (if available)
# re-fit original to get 'did'
y, X = dmatrices('log_migrants ~ did + C(pair) + C(year_month) - 1', df, return_type='dataframe')
mod_orig = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': df[pair_col]})
actual_did = mod_orig.params.get('did', np.nan)
print("Actual DID coef (original):", actual_did)

# plot histogram of placebo coefs and vertical line for actual
plt.figure(figsize=(8,4))
plt.hist(placebo_df['coef'], bins=12, alpha=0.8)
plt.axvline(actual_did, color='red', linestyle='--', linewidth=2, label='Actual CL->US DID')
plt.xlabel('Placebo DID coefficient (log scale)')
plt.ylabel('Count of placebo pairs')
plt.title('Placebo distribution (each control pair treated)')
plt.legend()
plt.tight_layout()
plt.show()

## In-time placebo (pretend treatment occurred at alternative dates)

Run the DID while moving the treatment date earlier or later (e.g., −12, −6, +6 months). For a valid design, fake treatment dates should produce null effects.

In [ ]:
# choose a grid of alternative offsets (months) from actual treatment_time
offsets = [-12, -6, 6, 12]    # earlier/later by months (adjust to your sample time unit)
time_results = []

for off in offsets:
    fake_t = treatment_time + off
    if fake_t <= df[time_col].min() or fake_t > df[time_col].max():
        print(f"Skipping fake_t={fake_t} (out of sample range)")
        continue
    tmp = df.copy()
    # define post relative to fake treatment time (but only treat the real treated pair as 'treated')
    tmp['fake_post'] = (tmp[time_col] >= fake_t).astype(int)
    tmp['fake_did'] = ((tmp[pair_col] == treated_pair_name).astype(int)) * tmp['fake_post']
    try:
        y_f, X_f = dmatrices('log_migrants ~ fake_did + C({}) + C({}) - 1'.format(pair_col, ym_col), tmp, return_type='dataframe')
        mod_f = sm.OLS(y_f, X_f).fit(cov_type='cluster', cov_kwds={'groups': tmp[pair_col]})
        coef_f = mod_f.params.get('fake_did', np.nan)
        se_f = mod_f.bse.get('fake_did', np.nan)
        pval_f = mod_f.pvalues.get('fake_did', np.nan)
        time_results.append({'offset': off, 'fake_t': fake_t, 'coef': coef_f, 'se': se_f, 'pval': pval_f})
    except Exception as e:
        time_results.append({'offset': off, 'fake_t': fake_t, 'coef': np.nan, 'se': np.nan, 'pval': np.nan})

pd_time = pd.DataFrame(time_results).dropna()
# print(pd_time)
pd_time

In [ ]:
# Plot fake-treatment coefficients with CIs vs actual
plt.figure(figsize=(8,4))
plt.errorbar(pd_time['offset'], pd_time['coef'], yerr=1.96*pd_time['se'], fmt='o', capsize=4, label='In-time placebo estimates')
plt.axhline(actual_coef, color='red', linestyle='--', label='Actual DID')
plt.axhline(0, color='gray', linestyle=':')
plt.xlabel('Offset from true treatment (months)')
plt.ylabel('Estimated coef (log scale)')
plt.title('In-time placebo: fake treatment dates')
plt.legend()
plt.show()

## Trimmed sample:
**Exclude potential anticipatory windows (drop pre-treatment months just before post)**

Sometimes agents anticipate policy and change behaviour in the few months immediately before post. Excluding those months is one way to test sensitivity.

Approach A — Drop a short window of observations (e.g., rel_time in [-w, -1]):

In [ ]:
# Choose anticipatory window length (in your time unit)
anticip_window = 3   # drop 3 months immediately before treatment (i.e. -3,-2,-1)
tmp = df.copy()
tmp['rel_time'] = tmp[time_col] - treatment_time
to_drop = tmp['rel_time'].between(-anticip_window, -1, inclusive='both')
print("Dropping rows with rel_time in [-{0}, -1]: {1} rows".format(anticip_window, to_drop.sum()))
tmp2 = tmp.loc[~to_drop].copy()

# Re-estimate event-study or simple DID on trimmed sample
y_trim, X_trim = dmatrices('log_migrants ~ did + C({}) + C({}) - 1'.format(pair_col, ym_col), tmp2, return_type='dataframe')
mod_trim = sm.OLS(y_trim, X_trim).fit(cov_type='cluster', cov_kwds={'groups': tmp2[pair_col]})
print("Trimmed-sample DID coef:", mod_trim.params.get('did', np.nan), "SE:", mod_trim.bse.get('did', np.nan))


## Event-study (leads and lags)

In [ ]:
month_fe_col = 'year_month'   # month fixed effects column
min_rel, max_rel = -12, 12
ref_period = -1

In [ ]:
# 1) relative time (clipped) and treated indicator
df = df.copy()
df['rel_time'] = df[time_col] - treatment_time
df['rel_time_clipped'] = df['rel_time'].clip(min_rel, max_rel)
df['is_treated_pair'] = (df[pair_col] == treated_pair).astype(int)

# 2) create relative-time dummies (they will be like 'rt_-3','rt_0','rt_4', but we'll sanitize later)
rel_dummies_full = pd.get_dummies(df['rel_time_clipped'].astype(pd.Int64Dtype()), prefix='rt')

# 3) drop the reference period dummy (if present), otherwise drop nearest available
ref_col = f'rt_{ref_period}'
if ref_col in rel_dummies_full.columns:
    rel_dummies_full = rel_dummies_full.drop(columns=[ref_col])
else:
    # choose available rel_time nearest to ref_period
    avail = []
    for c in rel_dummies_full.columns:
        m = re.match(r'^rt_(-?\d+)$', c)
        if m:
            avail.append(int(m.group(1)))
    if not avail:
        raise ValueError("No relative-time dummies created - check rel_time/clipping range.")
    nearest = min(avail, key=lambda x: abs(x - ref_period))
    drop_col = f'rt_{nearest}'
    rel_dummies_full = rel_dummies_full.drop(columns=[drop_col])
    print(f"Reference {ref_period} not present; dropped nearest available {drop_col} as reference.")

# 4) create sanitized event-interaction columns: evt_rt_m12 / evt_rt_0 / evt_rt_p5 etc.
def sanitize_rel_name(rel_int):
    if rel_int < 0:
        return f'evt_rt_m{abs(rel_int)}'
    elif rel_int > 0:
        return f'evt_rt_p{rel_int}'
    else:
        return 'evt_rt_0'

evt_cols = []
for col in rel_dummies_full.columns:
    m = re.match(r'^rt_(-?\d+)$', col)
    if not m:
        continue
    rel_int = int(m.group(1))
    safe_evt = sanitize_rel_name(rel_int)
    # align the dummy to df index and multiply by treated indicator
    df[safe_evt] = rel_dummies_full[col].fillna(0).astype(int) * df['is_treated_pair']
    evt_cols.append((rel_int, safe_evt))

# 5) sort evt_cols by numeric rel_time and extract sanitized names ordered
evt_cols = sorted(evt_cols, key=lambda x: x[0])
sanitized_evt_names = [name for _, name in evt_cols]
print("Sanitized event columns (ordered):", sanitized_evt_names)

# 6) build formula using sanitized names (Patsy-safe: no '-' in names)
formula = y_col + ' ~ ' + ' + '.join(sanitized_evt_names) + ' + C(' + pair_col + ') + C(' + month_fe_col + ') - 1'
print("Patsy-safe formula:\n", formula)

# 7) fit the model with cluster-robust SE by pair
y, X = dmatrices(formula, df, return_type='dataframe')
model = sm.OLS(y, X)
res = model.fit(cov_type='cluster', cov_kwds={'groups': df[pair_col]})
print(res.summary().tables[1])

# 8) extract coefficients and map sanitized names back to numeric rel_time for reporting
coef_ser = res.params.filter(like='evt_rt_')
se_ser = res.bse.filter(like='evt_rt_')

# map sanitized name -> numeric rel_time (works with our sanitize_rel_name)
def rel_from_sanitized(name):
    if name == 'evt_rt_0':
        return 0
    m_m = re.match(r'^evt_rt_m(\d+)$', name)
    if m_m:
        return -int(m_m.group(1))
    m_p = re.match(r'^evt_rt_p(\d+)$', name)
    if m_p:
        return int(m_p.group(1))
    raise ValueError(f"Unexpected evt name: {name}")

rel_times = [rel_from_sanitized(n) for n in coef_ser.index]
evt_df = pd.DataFrame({
    'rel_time': rel_times,
    'colname': coef_ser.index,
    'coef': coef_ser.values,
    'se': se_ser.values
}).sort_values('rel_time').reset_index(drop=True)
evt_df['ci_lower'] = evt_df['coef'] - 1.96*evt_df['se']
evt_df['ci_upper'] = evt_df['coef'] + 1.96*evt_df['se']

# show the event-study table
# print(evt_df[['rel_time','colname','coef','se','ci_lower','ci_upper']].to_string(index=False))
evt_df

In [ ]:
# 11. (Optional) Save results table for reporting
# evt_df.to_csv('event_study_coefficients.csv', index=False)
print("Saved event-study coefficients to 'event_study_coefficients.csv'")

In [ ]:
# 9) PRE-TREND JOINT TEST: test H0: all lead coefficients (rel_time < 0) = 0
lead_rows = evt_df[evt_df['rel_time'] < 0]
if lead_rows.shape[0] == 0:
    print("No lead coefficients (rel_time < 0) found in event window; nothing to test.")
else:
    param_names = res.params.index.tolist()
    R = np.zeros((len(lead_rows), len(param_names)))
    for i, col in enumerate(lead_rows['colname']):
        j = param_names.index(col)
        R[i, j] = 1.0
    ftest = res.f_test(R)
    print("Pre-trend joint F-test result:")
    print(ftest)   # contains F-stat, p-value, df_num/df_den

# evt_df is ready for plotting (rel_time vs coef with ci)

### Plot  

In [ ]:
# 10. Plot event-study coefficients with 95% CI
plt.figure(figsize=(9,5))
plt.errorbar(evt_df['rel_time'], evt_df['coef'], yerr=1.96*evt_df['se'],
             fmt='o-', capsize=4)
plt.axvline(-0.5, linestyle=':', linewidth=1)  # vertical line marking treatment between -1 and 0
plt.axhline(0, linestyle='--', linewidth=1)
plt.xlabel('Months relative to treatment (relative time)')
plt.ylabel('Estimated coefficient (log migrants)')
plt.title('Event-study: dynamic effects (reference = period {})'.format(ref_period))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9,5))

# 95% confidence band (shaded area)
plt.fill_between(
    evt_df['rel_time'],
    evt_df['ci_lower'],
    evt_df['ci_upper'],
    color='skyblue', alpha=0.3, label='95% CI'
)

# Coefficient line
plt.plot(
    evt_df['rel_time'],
    evt_df['coef'],
    color='blue', linewidth=2, label='Estimate'
)

# Horizontal and vertical reference lines
plt.axhline(0, color='black', linewidth=1, linestyle='--')
plt.axvline(0, color='red', linewidth=1.2, linestyle=':', label='Policy implementation')

# Labels and aesthetics
plt.xlabel('Months relative to policy implementation')
plt.ylabel('Effect on log migrants (vs. reference period)')
plt.title('Event-Study: Dynamic Effects of U.S. Policy on Outflows from Chile')
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

## Pair-specific linear trends

In [ ]:
# DID with pair-specific linear trends
import numpy as np
import pandas as pd
from patsy import dmatrices
import statsmodels.api as sm

df = BACI_df.copy()

# --- identify treated pair and treatment time (auto-detect) ---
treated_pair = 'CL->US'
if treated_pair not in df['pair'].unique():
    raise ValueError(f"{treated_pair} not found in df['pair']")

# treatment_time: first time_index where treated pair has post==1 (assumes 'post' exists)
mask_treated_post = (df['pair'] == treated_pair) & (df['post'] == 1)
if mask_treated_post.any():
    treatment_time = int(df.loc[mask_treated_post, 'time_index'].min())
else:
    raise ValueError("Cannot auto-detect treatment_time: no rows with pair==treated_pair and post==1")

print("Detected treatment_time =", treatment_time)

# ensure time_index numeric
df['time_index'] = df['time_index'].astype(int)

# --- baseline DID (your original specification) ---
formula_base = 'log_migrants ~ did + C(pair) + C(year_month) - 1'
y_base, X_base = dmatrices(formula_base, df, return_type='dataframe')
mod_base = sm.OLS(y_base, X_base).fit(cov_type='cluster', cov_kwds={'groups': df['pair']})
coef_base = mod_base.params.get('did', np.nan)
se_base = mod_base.bse.get('did', np.nan)

# --- DID with pair-specific linear trends ---
# interaction C(pair):time_index creates a linear time trend for each pair
formula_trend = 'log_migrants ~ did + C(pair) + C(year_month) + C(pair):time_index - 1'
y_tr, X_tr = dmatrices(formula_trend, df, return_type='dataframe')
mod_tr = sm.OLS(y_tr, X_tr).fit(cov_type='cluster', cov_kwds={'groups': df['pair']})
coef_tr = mod_tr.params.get('did', np.nan)
se_tr = mod_tr.bse.get('did', np.nan)

# Helper to convert log-coef to pct
def pct_from_log(beta):
    return (np.exp(beta) - 1) * 100

print("\n=== Baseline DID ===")
print(f"Coef (log): {coef_base:.6f}, SE (clustered by pair): {se_base:.6f}")
print(f"Approx % change: {pct_from_log(coef_base):.1f}%")

print("\n=== DID with pair-specific linear trends ===")
print(f"Coef (log): {coef_tr:.6f}, SE (clustered by pair): {se_tr:.6f}")
print(f"Approx % change: {pct_from_log(coef_tr):.1f}%")

# Optional: significance stars
def star(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.1: return '*'
    return ''
print("\nSignificance (with pair-trends):", star(mod_tr.pvalues.get('did', 1.0)))


## Wide in-time placebo grid (fake treatment dates in pre-period)

This loops over plausible fake treatment months in the pre-treatment period (every month before the true treatment_time), estimates DID as if treatment happened at fake_t, and returns the distribution of fake-date coefficients. It then shows how many fake-date coefs are ≥ the actual DID (one-sided upper empirical p), and plots a histogram with the real DID marked.

In [ ]:
# In-time placebo grid (loop fake treatment dates)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from patsy import dmatrices
import statsmodels.api as sm

df = BACI_df.copy()
df['time_index'] = df['time_index'].astype(int)
treated_pair = 'CL->US'
treatment_time = int(df.loc[(df['pair']==treated_pair)&(df['post']==1),'time_index'].min())
print("Using treatment_time =", treatment_time)

# choose candidate fake treatment times: all months strictly before real treatment_time
fake_times = sorted(df['time_index'].unique())
fake_times = [t for t in fake_times if t < treatment_time]

placebo_results = []
for fake_t in fake_times:
    tmp = df.copy()
    # define fake 'post' and fake 'did' (treated pair has placebo_treatment from fake_t onward)
    tmp['fake_post'] = (tmp['time_index'] >= fake_t).astype(int)
    tmp['fake_treated'] = (tmp['pair'] == treated_pair).astype(int)
    tmp['fake_did'] = tmp['fake_post'] * tmp['fake_treated']
    try:
        y, X = dmatrices('log_migrants ~ fake_did + C(pair) + C(year_month) - 1', tmp, return_type='dataframe')
        mod = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': tmp['pair']})
        coef = mod.params.get('fake_did', np.nan)
        se = mod.bse.get('fake_did', np.nan)
        pval = mod.pvalues.get('fake_did', np.nan)
        placebo_results.append({'fake_t': fake_t, 'coef': float(coef), 'se': float(se), 'pval': float(pval)})
    except Exception as e:
        # in case of collinearity or too-few observations for some fake_t
        placebo_results.append({'fake_t': fake_t, 'coef': np.nan, 'se': np.nan, 'pval': np.nan})

placebo_df = pd.DataFrame(placebo_results).dropna(subset=['coef']).reset_index(drop=True)

# Actual DID coefficient (from baseline)
y_base, X_base = dmatrices('log_migrants ~ did + C(pair) + C(year_month) - 1', df, return_type='dataframe')
mod_base = sm.OLS(y_base, X_base).fit(cov_type='cluster', cov_kwds={'groups': df['pair']})
actual_coef = float(mod_base.params.get('did', np.nan))

# empirical p-values
N_placebos = len(placebo_df)
n_ge = (placebo_df['coef'] >= actual_coef).sum()  # how many fake coefs >= actual
# Add one to numerator & denom for conservative p (including actual) often done: (count+1)/(N+1)
empirical_p_one_sided = (n_ge + 1) / (N_placebos + 1)
empirical_p_two_sided = ( ( (placebo_df['coef'] >= abs(actual_coef)).sum() + (placebo_df['coef'] <= -abs(actual_coef)).sum() ) + 1 ) / (N_placebos + 1)

print("\n=== In-time placebo grid summary ===")
print(f"N_placebos = {N_placebos}")
print(f"Actual DID coef = {actual_coef:.6f}")
print(f"Number of fake dates with coef >= actual: {n_ge}")
print(f"Empirical p-value (one-sided upper, conservative) = {empirical_p_one_sided:.3f}")
print(f"Empirical p-value (two-sided, conservative) = {empirical_p_two_sided:.3f}")

# Quick table: first few fake_t results
print("\nFirst 10 placebo results (fake_t, coef, se, pval):")
# print(placebo_df[['fake_t','coef','se','pval']].head(10).to_string(index=False))
placebo_df

In [ ]:
# Plot histogram
plt.figure(figsize=(8,4))
plt.hist(placebo_df['coef'], bins=15, alpha=0.8)
plt.axvline(actual_coef, color='red', linestyle='--', linewidth=2, label=f'Actual DID = {actual_coef:.3f}')
plt.xlabel('Placebo DID coefficient (log scale)')
plt.ylabel('Count of fake treatment dates')
plt.title('In-time placebo grid: distribution of fake-date DID coefs')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Optional: time-series plot of placebo coefficients vs fake_t for pattern
plt.figure(figsize=(9,4))
plt.plot(placebo_df['fake_t'], placebo_df['coef'], marker='o', linestyle='-')
plt.axhline(actual_coef, color='red', linestyle='--', label='Actual DID')
plt.xlabel('Fake treatment time_index')
plt.ylabel('Estimated fake-date DID coef (log)')
plt.title('Fake-date DID coefficients over time (pre-period)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# assumes placebo_df exists (from your placebo loop) OR re-run quickly to get placebo_df
# if you don't have placebo_df, re-run the in-time placebo loop from earlier and produce placebo_df

# Map fake_t -> representative year_month (you likely have year_month per time_index)
time_map = df[['time_index','year_month']].drop_duplicates().set_index('time_index')['year_month'].to_dict()

# Suppose placebo_df is your DataFrame with columns ['fake_t','coef','se','pval']
# If not present, re-create as in the code I gave you earlier
placebo_df['year_month'] = placebo_df['fake_t'].map(time_map)

# show the fake dates with coef >= actual
actual_coef = float(mod_base.params.get('did'))
big_fakes = placebo_df[placebo_df['coef'] >= actual_coef].sort_values('coef', ascending=False)
print("Fake dates with coef >= actual (fake_t, year_month, coef, se, pval):")
print(big_fakes[['fake_t','year_month','coef','se','pval']].to_string(index=False))

# show top 10 largest fake coefs (to see if a few extreme months dominate)
print("\nTop 10 largest fake-date coefficients:")
print(placebo_df.sort_values('coef', ascending=False).head(10)[['fake_t','year_month','coef','se','pval']].to_string(index=False))

In [ ]:
# Event-study WITH pair-specific linear trends — safe column names
import re
import numpy as np
import pandas as pd
from patsy import dmatrices
import statsmodels.api as sm
import matplotlib.pyplot as plt

df = BACI_df.copy()
treated_pair = 'CL->US'
treatment_time = int(df.loc[(df['pair']==treated_pair)&(df['post']==1),'time_index'].min())

# 1) create rel_time and clipped range
df['rel_time'] = df['time_index'] - treatment_time
min_period, max_period = -12, 12
df['rel_time_clipped'] = df['rel_time'].clip(min_period, max_period)

# 2) create rel dummies with *safe* names: mXX for negative, pXX for positive, zero -> z0
def safe_rel_name(x):
    if x < 0:
        return f'rt_m{abs(int(x))}'
    if x == 0:
        return 'rt_z0'
    return f'rt_p{int(x)}'

# make a column with safe rel labels for get_dummies
df['rel_label'] = df['rel_time_clipped'].apply(safe_rel_name)

# create dummies
rel_dummies = pd.get_dummies(df['rel_label'], prefix='evt_rt')
# drop reference period (e.g., rt_m1 -> 'rt_m1' corresponds to reference -1; adjust if you used -1)
ref_label = safe_rel_name(-1)  # ref = -1
ref_col = f'evt_rt_{ref_label}'
if ref_col in rel_dummies.columns:
    rel_dummies = rel_dummies.drop(columns=[ref_col])

# attach dummies to df
for col in rel_dummies.columns:
    df[col] = rel_dummies[col].values

# 3) create treated indicator and interacted event dummies (these will be named evt_rt_<label> already, but we want them to be multiplied by treated)
is_treated = (df['pair'] == treated_pair).astype(int)
evt_cols = []
for col in rel_dummies.columns:
    newcol = f'evt_{col}'         # e.g. evt_evt_rt_rt_p1  -> we'll rename below to simpler
    df[newcol] = df[col] * is_treated
    # simplify name: drop double prefixes, keep 'evt_rt_p1' etc.
    # final name will be like 'evt_rt_p1' or 'evt_rt_m12' etc.
    safe_name = col.replace('evt_rt_', 'evt_rt_')  # already good, but ensure consistent
    # rename created newcol to 'evt_' + safe_label (ensure uniqueness)
    final_name = 'evt_' + col.replace('evt_rt_', '')
    df.rename(columns={newcol: final_name}, inplace=True)
    evt_cols.append(final_name)

# double-check evt_cols look fine
print("Event dummy columns (example):", evt_cols[:8])

# 4) Build formula with safe names. Include pair-specific linear trend via C(pair):time_index
# Use -1 to omit intercept because C(pair) will absorb it
formula_evt_trend = 'log_migrants ~ ' + ' + '.join(evt_cols) + ' + C(pair) + C(year_month) + C(pair):time_index - 1'
print("Formula length, number of evt terms:", len(evt_cols))

# 5) Fit model (cluster SE by pair)
y_evt, X_evt = dmatrices(formula_evt_trend, df, return_type='dataframe')
mod_evt_trend = sm.OLS(y_evt, X_evt).fit(cov_type='cluster', cov_kwds={'groups': df['pair']})

# 6) Extract event coefficients and map rel_time
coef_ser = mod_evt_trend.params.filter(like='evt_rt_')
se_ser = mod_evt_trend.bse.filter(like='evt_rt_')

# Recover rel_time numeric from column names
def rel_from_col(cname):
    # expects 'evt_rt_rt_m12' or 'evt_rt_m12' or 'evt_rt_p1' depending on naming above
    m = re.search(r'evt_rt_?m(\d+)', cname)
    if m:
        return -int(m.group(1))
    z = re.search(r'evt_rt_?z0', cname)
    if z:
        return 0
    p = re.search(r'evt_rt_?p(\d+)', cname)
    if p:
        return int(p.group(1))
    # fallback: try to parse digits
    nums = re.findall(r'-?\d+', cname)
    return int(nums[-1]) if nums else np.nan

evt_table = pd.DataFrame({
    'colname': coef_ser.index,
    'coef': coef_ser.values,
    'se': se_ser.values
})
evt_table['rel_time'] = evt_table['colname'].apply(rel_from_col)
evt_table = evt_table.sort_values('rel_time').reset_index(drop=True)

# 7) print table (rel_time, coef, se, 95% CI)
evt_table['ci_lower'] = evt_table['coef'] - 1.96 * evt_table['se']
evt_table['ci_upper'] = evt_table['coef'] + 1.96 * evt_table['se']
print(evt_table[['rel_time','colname','coef','se','ci_lower','ci_upper']].to_string(index=False))

# 8) plot event-study coefficients
plt.figure(figsize=(9,5))
plt.errorbar(evt_table['rel_time'], evt_table['coef'], yerr=1.96*evt_table['se'], fmt='o-', capsize=4)
plt.axvline(-0.5, color='k', linestyle=':', label='Treatment')
plt.axhline(0, color='grey', linestyle='--')
plt.xlabel('Relative time (months) to treatment')
plt.ylabel('Estimated coefficient (log scale)')
plt.title('Event-study (with pair-specific linear trends)')
plt.tight_layout()
plt.show()


In [ ]:
# Joint test: all pre-period event coefficients = 0
# This uses statsmodels' wald_test (which handles clustered var-cov if provided via 'cov_p' argument)
import numpy as np

# get list of pre-event coef names from mod_evt_trend
pre_names = [name for name in mod_evt_trend.params.index if name.startswith('evt_rt_') and ('m' in name or 'z0' in name) and not name.endswith('p1')]
# alternatively, filter on rel_time < 0 if you have evt_table
# form the restriction matrix for Wald test: R * beta = 0
R = np.zeros((len(pre_names), len(mod_evt_trend.params)))
param_names = list(mod_evt_trend.params.index)
for i, pname in enumerate(pre_names):
    j = param_names.index(pname)
    R[i, j] = 1.0

# covariance of params: use cluster-robust cov from the model
cov = mod_evt_trend.cov_params()  # note: this is the clustered cov because you fit with cov_type='cluster'
wald_res = mod_evt_trend.wald_test(R)
print("Joint pre-trend test (Wald):")
print(wald_res)  # prints F-stat or Chi2 and p-value

# If you want the numeric F and p explicitly:
stat = wald_res.statistic
pval = wald_res.pvalue
print(f"Joint test statistic = {stat}, p-value = {pval}")


In [ ]:
# Create quadratic time term and add pair-specific quadratic trends
df['time_index'] = df['time_index'].astype(int)
df['time_sq'] = df['time_index'] ** 2

# Refit event-study with pair linear + quadratic trends
# Build formula: event dummies (evt_cols) + C(pair) + C(year_month) + C(pair):time_index + C(pair):time_sq - 1
trend_terms = ' + '.join(evt_cols) + ' + C(pair) + C(year_month) + C(pair):time_index + C(pair):time_sq - 1'
y_evt2, X_evt2 = dmatrices('log_migrants ~ ' + trend_terms, df, return_type='dataframe')
mod_evt_trend2 = sm.OLS(y_evt2, X_evt2).fit(cov_type='cluster', cov_kwds={'groups': df['pair']})

# Joint pre-trend test (pre-period evt names detection)
pre_names = [name for name in mod_evt_trend2.params.index if name.startswith('evt_rt_') and ('m' in name or 'z0' in name)]
R = np.zeros((len(pre_names), len(mod_evt_trend2.params)))
param_names = list(mod_evt_trend2.params.index)
for i, pname in enumerate(pre_names):
    j = param_names.index(pname)
    R[i, j] = 1.0

wald_res2 = mod_evt_trend2.wald_test(R)
print("Quadratic-trend model joint pre-test:", wald_res2)


# Result

In [ ]:
# === plot_suite.py ===
# Copy-paste this into your notebook or script. Produces Figures 1-4 and Appendix A1 as PNGs.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from patsy import dmatrices

# --------- User settings: change if needed ----------
df = BACI_df.copy()                       # your dataframe
treated_pair = 'CL->US'                   # treated pair name
pair_col = 'pair'
time_col = 'time_index'
ym_col = 'year_month'
y_col = 'log_migrants'
treatment_time = int(df.loc[(df[pair_col]==treated_pair)&(df['post']==1), time_col].min())
out_prefix = 'figure_'                     # saved file prefix
# ---------------------------------------------------

# Basic checks
assert treated_pair in df[pair_col].unique(), "Treated pair not found"
assert time_col in df.columns and y_col in df.columns, "Required columns missing"

# Helper to compute event-study coefficients (returns DataFrame with rel_time, coef, se, ci)
def compute_event_study(df_in, treated_pair, treatment_time, min_period=-12, max_period=12, with_pair_trends=False):
    df2 = df_in.copy()
    df2['rel_time'] = df2[time_col] - treatment_time
    df2['rel_time_clipped'] = df2['rel_time'].clip(min_period, max_period)
    # make safe labels for rel_time
    def safe_rel_name(x):
        if x < 0:
            return f'rt_m{abs(int(x))}'
        if x == 0:
            return 'rt_z0'
        return f'rt_p{int(x)}'
    df2['rel_label'] = df2['rel_time_clipped'].apply(safe_rel_name)
    rel_dummies = pd.get_dummies(df2['rel_label'], prefix='rt')
    # drop ref period (-1)
    ref_label = safe_rel_name(-1)
    ref_col = f'rt_{ref_label}'
    if ref_col in rel_dummies.columns:
        rel_dummies = rel_dummies.drop(columns=[ref_col])
    # attach dummies
    for col in rel_dummies.columns:
        df2[col] = rel_dummies[col].values
    # interacted dummies: only active for treated pair
    df2['is_treated_pair'] = (df2[pair_col] == treated_pair).astype(int)
    evt_cols = []
    for col in rel_dummies.columns:
        final = f'evt_{col}'  # e.g. evt_rt_rt_p1 -> simplified by construction
        df2[final] = df2[col] * df2['is_treated_pair']
        evt_cols.append(final)
    # Build formula
    # -1 to omit intercept because C(pair) will absorb it
    base_rhs = ' + '.join(evt_cols) + ' + C(' + pair_col + ') + C(' + ym_col + ') - 1'
    if with_pair_trends:
        # add pair x linear time_index
        formula = f"{y_col} ~ {base_rhs} + C({pair_col}):{time_col}"
    else:
        formula = f"{y_col} ~ {base_rhs}"
    # design matrices and OLS
    y, X = dmatrices(formula, df2, return_type='dataframe')
    model = sm.OLS(y, X)
    res = model.fit(cov_type='cluster', cov_kwds={'groups': df2[pair_col]})
    # extract evt coefficients and map rel_time
    coef = res.params.filter(like='evt_rt_')
    se = res.bse.filter(like='evt_rt_')
    # parse rel_time numeric from column names
    def rel_from_col(cname):
        # expects 'evt_rt_rt_m12' or 'evt_rt_rt_p1' or 'evt_rt_rt_z0' depending on naming
        s = cname
        # search for mNN, pNN, z0
        if '_m' in s:
            return -int(s.split('_m')[-1])
        if '_p' in s:
            return int(s.split('_p')[-1])
        if 'z0' in s:
            return 0
        # fallback
        nums = [int(x) for x in re.findall(r'-?\d+', s)]
        return nums[-1] if nums else np.nan
    evt_df = pd.DataFrame({
        'colname': coef.index,
        'coef': coef.values,
        'se': se.values
    })
    # map rel_time robustly by extracting digits at end
    def parse_rel(c):
        # try patterns
        c2 = c.replace('evt_rt_rt_', '').replace('evt_rt_', '')
        if c2.startswith('m'):
            return -int(c2[1:])
        if c2.startswith('p'):
            return int(c2[1:])
        if c2 == 'z0':
            return 0
        # fallback numeric extraction
        import re
        nums = re.findall(r'-?\d+', c2)
        return int(nums[-1]) if nums else np.nan
    evt_df['rel_time'] = evt_df['colname'].apply(parse_rel)
    evt_df = evt_df.sort_values('rel_time').reset_index(drop=True)
    evt_df['ci_lower'] = evt_df['coef'] - 1.96 * evt_df['se']
    evt_df['ci_upper'] = evt_df['coef'] + 1.96 * evt_df['se']
    return res, evt_df

# ------- Figure 1: Parallel-means -------
def fig_parallel_means(df, treated_pair, time_col, y_col, outname):
    dfm = df.copy()
    dfm['group'] = np.where(dfm[pair_col] == treated_pair, 'Treated (CL->US)', 'Control (all)')
    mean_by_time = dfm.groupby([time_col, 'group'])[y_col].mean().reset_index()
    plt.figure(figsize=(10,5))
    for gname, g in mean_by_time.groupby('group'):
        plt.plot(g[time_col], g[y_col], label=gname, linewidth=2 if gname.startswith('Treated') else 1.5, linestyle='--' if gname.startswith('Control') else '-')
    plt.axvline(treatment_time-0.5, linestyle=':', linewidth=1)
    plt.xlabel('Time index')
    plt.ylabel('Average log migrants')
    plt.title('Average log migrants: Treated (CL→US) vs Pooled controls')
    plt.legend()
    plt.tight_layout()
    plt.savefig(outname, dpi=200)
    plt.close()

# ------- Figure 2: Spaghetti plot -------
def fig_spaghetti(df, treated_pair, time_col, y_col, outname):
    plt.figure(figsize=(11,6))
    for p in df[pair_col].unique():
        s = df[df[pair_col]==p].sort_values(time_col)
        if p == treated_pair:
            plt.plot(s[time_col], s[y_col], linewidth=2.5, label=p)
        else:
            plt.plot(s[time_col], s[y_col], alpha=0.3)
    plt.axvline(treatment_time-0.5, linestyle=':', linewidth=1)
    plt.xlabel('Time index')
    plt.ylabel('log migrants')
    plt.title('Spaghetti plot: each pair time-series (treated highlighted)')
    plt.legend(loc='upper left')
    plt.tight_layout()
    plt.savefig(outname, dpi=200)
    plt.close()

# ------- Figure 3: Event-study (two panels) -------
def fig_event_study(df, treated_pair, treatment_time, outname_panel):
    # compute both versions
    res_ntrend, evt_ntrend = compute_event_study(df, treated_pair, treatment_time, with_pair_trends=False)
    res_trend, evt_trend = compute_event_study(df, treated_pair, treatment_time, with_pair_trends=True)
    # plot side-by-side
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14,5), sharey=True)
    # Panel A: no pair trends
    axes[0].errorbar(evt_ntrend['rel_time'], evt_ntrend['coef'], yerr=1.96*evt_ntrend['se'], fmt='o-', capsize=4)
    axes[0].axvline(-0.5, linestyle=':', linewidth=1)
    axes[0].axhline(0, linestyle='--', linewidth=1)
    axes[0].set_title('Event-study (no pair trends)')
    axes[0].set_xlabel('Relative months')
    axes[0].set_ylabel('Coeff (log)')
    # Panel B: with pair trends
    axes[1].errorbar(evt_trend['rel_time'], evt_trend['coef'], yerr=1.96*evt_trend['se'], fmt='o-', capsize=4)
    axes[1].axvline(-0.5, linestyle=':', linewidth=1)
    axes[1].axhline(0, linestyle='--', linewidth=1)
    axes[1].set_title('Event-study (with pair-specific linear trends)')
    axes[1].set_xlabel('Relative months')
    plt.suptitle('Event-study: dynamic DID (reference period = -1)')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(outname_panel, dpi=200)
    plt.close()
    return (res_ntrend, evt_ntrend), (res_trend, evt_trend)

# ------- Figure 4: In-space permutation histogram -------
def fig_in_space_permutation(df, treated_pair, treatment_time, outname):
    # other pairs
    pairs = [p for p in df[pair_col].unique() if p != treated_pair]
    placebo_coefs = []
    for donor in pairs:
        tmp = df.copy()
        tmp['placebo_treated'] = (tmp[pair_col] == donor).astype(int)
        tmp['placebo_post'] = (tmp[time_col] >= treatment_time).astype(int)
        tmp['placebo_did'] = tmp['placebo_treated'] * tmp['placebo_post']
        try:
            y, X = dmatrices('log_migrants ~ placebo_did + C(pair) + C(year_month) - 1', tmp, return_type='dataframe')
            mod = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': tmp[pair_col]})
            placebo_coefs.append(mod.params.get('placebo_did', np.nan))
        except Exception:
            placebo_coefs.append(np.nan)
    placebo_coefs = np.array([c for c in placebo_coefs if not pd.isna(c)])
    # actual
    yb, Xb = dmatrices('log_migrants ~ did + C(pair) + C(year_month) - 1', df, return_type='dataframe')
    modb = sm.OLS(yb, Xb).fit(cov_type='cluster', cov_kwds={'groups': df[pair_col]})
    actual = float(modb.params.get('did'))
    # plot
    plt.figure(figsize=(8,4))
    plt.hist(placebo_coefs, bins=12, alpha=0.8)
    plt.axvline(actual, linestyle='--', linewidth=2)
    plt.xlabel('Placebo DID coef (log scale)')
    plt.ylabel('Count')
    plt.title('In-space permutation: placebo distribution (each control pair treated)')
    plt.tight_layout()
    plt.savefig(outname, dpi=200)
    plt.close()

# # ------- Appendix Fig A1: In-time placebo grid -------
# def fig_in_time_placebo(df, treated_pair, treatment_time, outname_hist, outname_line):
#     fake_times = sorted(df[time_col].unique())
#     fake_times = [t for t in fake_times if t < treatment_time]  # pre-period only
#     results = []
#     for ft in fake_times:
#         tmp = df.copy()
#         tmp['fake_post'] = (tmp[time_col] >= ft).astype(int)
#         tmp['fake_treated'] = (tmp[pair_col] == treated_pair).astype(int)
#         tmp['fake_did'] = tmp['fake_post'] * tmp['fake_treated']
#         try:
#             y, X = dmatrices('log_migrants ~ fake_did + C(pair) + C(year_month) - 1', tmp, return_type='dataframe')
#             mod = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': tmp[pair_col]})
#             results.append({'fake_t': ft, 'coef': float(mod.params.get('fake_did', np.nan)), 'se': float(mod.bse.get('fake_did', np.nan))})
#         except Exception:
#             results.append({'fake_t': ft, 'coef': np.nan, 'se': np.nan})
#     resdf = pd.DataFrame(results).dropna(subset=['coef']).reset_index(drop=True)
#     # histogram
#     plt.figure(figsize=(8,4))
#     plt.hist(resdf['coef'], bins=20, alpha=0.8)
#     plt.title('In-time placebo: distribution of fake-date DID coefficients')
#     plt.xlabel('Fake-date coef (log)')
#     plt.tight_layout()
#     plt.savefig(outname_hist, dpi=200)
#     plt.close()
    
#     # line over time
#     plt.figure(figsize=(9,4))
#     plt.plot(resdf['fake_t'], resdf['coef'], marker='o', linestyle='-')
#     plt.axhline(y=float(dmatrices('log_migrants ~ did + C(pair) + C(year_month) - 1', df, return_type='dataframe')[0].mean()), linestyle='--')  # placeholder if needed
#     plt.title('In-time placebo coefficients over pre-period dates')
#     plt.xlabel('Fake treatment time_index')
#     plt.ylabel('Estimated fake-date DID coef (log)')
#     plt.tight_layout()
#     plt.savefig(outname_line, dpi=200)
#     plt.close()
#     return resdf


# Corrected version of fig_in_time_placebo with actual DID horizontal line and nice labels
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from patsy import dmatrices
import statsmodels.api as sm

def fig_in_time_placebo(df, treated_pair, time_col, ym_col, treatment_time, outname_hist, outname_line):
    # pre-period fake times
    fake_times = sorted(df[time_col].unique())
    fake_times = [t for t in fake_times if t < treatment_time]
    results = []
    for ft in fake_times:
        tmp = df.copy()
        tmp['fake_post'] = (tmp[time_col] >= ft).astype(int)
        tmp['fake_treated'] = (tmp['pair'] == treated_pair).astype(int)
        tmp['fake_did'] = tmp['fake_post'] * tmp['fake_treated']
        try:
            y, X = dmatrices('log_migrants ~ fake_did + C(pair) + C(year_month) - 1', tmp, return_type='dataframe')
            mod = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': tmp['pair']})
            coef = float(mod.params.get('fake_did', np.nan))
            se = float(mod.bse.get('fake_did', np.nan))
            results.append({'fake_t': ft, 'coef': coef, 'se': se})
        except Exception:
            results.append({'fake_t': ft, 'coef': np.nan, 'se': np.nan})
    resdf = pd.DataFrame(results).dropna(subset=['coef']).reset_index(drop=True)

    # compute actual DID coef (baseline)
    y_act, X_act = dmatrices('log_migrants ~ did + C(pair) + C(year_month) - 1', df, return_type='dataframe')
    mod_act = sm.OLS(y_act, X_act).fit(cov_type='cluster', cov_kwds={'groups': df['pair']})
    actual_coef = float(mod_act.params.get('did', np.nan))

    # Histogram of fake-date coefs
    plt.figure(figsize=(8,4))
    plt.hist(resdf['coef'], bins=20, alpha=0.8)
    plt.axvline(actual_coef, color='red', linestyle='--', linewidth=2, label=f'Actual DID = {actual_coef:.3f}')
    plt.xlabel('Fake-date coef (log)')
    plt.title('In-time placebo: distribution of fake-date DID coefficients')
    plt.legend()
    plt.tight_layout()
    plt.savefig(outname_hist, dpi=200)
    plt.close()

    # Map fake_t to year_month strings for x-axis labels
    time_to_ym = df[[time_col, ym_col]].drop_duplicates().set_index(time_col)[ym_col].to_dict()
    resdf['year_month'] = resdf['fake_t'].map(time_to_ym)

    # mark which fake months exceed or equal actual
    resdf['exceeds_actual'] = resdf['coef'] >= actual_coef

    # Line plot across fake times, highlight exceeders
    plt.figure(figsize=(10,4))
    plt.plot(resdf['fake_t'], resdf['coef'], marker='o', linestyle='-', label='Fake-date coef')
    # highlight the exceeders
    exceed = resdf[resdf['exceeds_actual']]
    if not exceed.empty:
        plt.scatter(exceed['fake_t'], exceed['coef'], color='red', s=80, label='Coef >= actual', zorder=5)
        for _, r in exceed.iterrows():
            plt.text(r['fake_t'], r['coef'] + 0.03, r['year_month'], color='red', ha='center', fontsize=8)
    # actual DID horizontal line
    plt.axhline(y=actual_coef, color='red', linestyle='--', linewidth=1.5, label=f'Actual DID = {actual_coef:.3f}')
    # vertical line showing the true treatment_time (optional)
    plt.axvline(x=treatment_time, color='grey', linestyle=':', linewidth=1)
    # nicer x-ticks: choose a subset to avoid crowding
    ticks = resdf['fake_t'].tolist()
    # map labels; rotate to fit
    labels = [time_to_ym[t] for t in ticks]
    plt.xticks(ticks, labels, rotation=45, fontsize=8)
    plt.xlabel('Fake treatment month (year-month)')
    plt.ylabel('Estimated fake-date DID coef (log)')
    plt.title('In-time placebo coefficients over pre-period dates')
    plt.legend(loc='best')
    plt.tight_layout()
    plt.savefig(outname_line, dpi=200)
    plt.close()

    return resdf, actual_coef

# Example usage:
# resdf, actual_coef = fig_in_time_placebo(BACI_df, 'CL->US', 'time_index', 'year_month', treatment_time=28,
#                                         outname_hist='figA1_intime_hist_fixed.png',
#                                         outname_line='figA1_intime_line_fixed.png')


In [ ]:
# ------------------- Run everything and save -------------------
fig_parallel_means(df, treated_pair, time_col, y_col, out_prefix + 'fig1_parallel_means.png')
fig_spaghetti(df, treated_pair, time_col, y_col, out_prefix + 'fig2_spaghetti.png')
(evt_n, evt_df_n), (evt_t, evt_df_t) = fig_event_study(df, treated_pair, treatment_time, out_prefix + 'fig3_event_study.png')
fig_in_space_permutation(df, treated_pair, treatment_time, out_prefix + 'fig4_inspace_perm.png')
# res_in_time = fig_in_time_placebo(df, treated_pair, treatment_time, out_prefix + 'figA1_intime_hist.png', out_prefix + 'figA1_intime_line.png')

print("Saved figures:")
print(out_prefix + 'fig1_parallel_means.png')
print(out_prefix + 'fig2_spaghetti.png')
print(out_prefix + 'fig3_event_study.png')
print(out_prefix + 'fig4_inspace_perm.png')
# print(out_prefix + 'figA1_intime_hist.png', out_prefix + 'figA1_intime_line.png')

In [ ]:
resdf, actual_coef = fig_in_time_placebo(BACI_df, 'CL->US', 'time_index', 'year_month', treatment_time=28,
                                        outname_hist='figA1_intime_hist_fixed.png',
                                        outname_line='figA1_intime_line_fixed.png')


In [ ]:
import numpy as np
import pandas as pd
from patsy import dmatrices
import statsmodels.api as sm

df = BACI_df.copy()
treated_pair = "CL->US"

# ----- 1.  baseline DID -----
y_b, X_b = dmatrices('log_migrants ~ did + C(pair) + C(year_month) - 1',
                     df, return_type='dataframe')
mod_b = sm.OLS(y_b, X_b).fit(cov_type='cluster', cov_kwds={'groups': df['pair']})

# ----- 2.  DID + pair-specific linear trends -----
y_t, X_t = dmatrices('log_migrants ~ did + C(pair) + C(year_month) + C(pair):time_index - 1',
                     df, return_type='dataframe')
mod_t = sm.OLS(y_t, X_t).fit(cov_type='cluster', cov_kwds={'groups': df['pair']})

# ----- 3.  Trimmed sample (drop rel_time ∈ [-3,-1]) -----
treatment_time = int(df.loc[(df['pair']==treated_pair)&(df['post']==1),'time_index'].min())
df_trim = df.loc[~df['time_index'].between(treatment_time-3, treatment_time-1)].copy()
y_trim, X_trim = dmatrices('log_migrants ~ did + C(pair) + C(year_month) - 1',
                           df_trim, return_type='dataframe')
mod_trim = sm.OLS(y_trim, X_trim).fit(cov_type='cluster',
                                      cov_kwds={'groups': df_trim['pair']})

# ----- 4.  Function to summarise results -----
def summarize_model(model, label):
    b = model.params.get('did', np.nan)
    se = model.bse.get('did', np.nan)
    z = b / se
    p = model.pvalues.get('did', np.nan)
    ci_low, ci_high = b - 1.96*se, b + 1.96*se
    pct = (np.exp(b)-1)*100
    pct_low, pct_high = (np.exp(ci_low)-1)*100, (np.exp(ci_high)-1)*100
    return {
        "Specification": label,
        "Coef (log)": round(b,6),
        "SE": round(se,6),
        "z": round(z,2),
        "p-value": p,
        "95% CI (log)": f"[{ci_low:.3f}, {ci_high:.3f}]",
        "Approx % change": f"{pct:.1f}%",
        "95% CI (% change)": f"[{pct_low:.1f}%, {pct_high:.1f}%]"
    }

rows = [
    summarize_model(mod_b, "Baseline DID (pair FE + month FE)"),
    summarize_model(mod_t, "DID + pair-specific linear trends"),
    summarize_model(mod_trim, "Trimmed sample (drop [-3,-1])")
]

table1_df = pd.DataFrame(rows).round(4).T
table1_df

In [ ]:
# Table A1: Extended robustness table
# Paste and run this in your notebook (after the usual imports)
import numpy as np
import pandas as pd
from patsy import dmatrices
import statsmodels.api as sm
import math

df = BACI_df.copy()   # your dataframe
treated_pair = 'CL->US'
pair_col = 'pair'
time_col = 'time_index'
ym_col = 'year_month'
y_col = 'log_migrants'
treatment_time = int(df.loc[(df[pair_col]==treated_pair)&(df['post']==1), time_col].min())

# Helper: run regression and return summary metrics
def run_reg(formula, df_local, cluster_col=pair_col):
    y, X = dmatrices(formula, df_local, return_type='dataframe')
    mod = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': df_local[cluster_col]})
    coef = float(mod.params.iloc[0]) if len(mod.params)>0 and 'did' in mod.params.index else \
           float(mod.params.get('did', np.nan)) if 'did' in mod.params.index else float(np.nan)
    # safer: get named param if present
    param_name = None
    # find the treatment param name among possible names (did, fake_did, placebo_did)
    for cand in ['did','placebo_did','fake_did']:
        if cand in mod.params.index:
            param_name = cand
            break
    if param_name is None:
        # fallback: choose first numeric column that is not a FE or time FE
        param_name = [c for c in mod.params.index if c not in X.columns[X.columns.str.startswith('C(')] and c!='Intercept']
        param_name = param_name[0] if len(param_name)>0 else None
    if param_name is None:
        raise ValueError("Could not find treatment parameter in model. Adjust formula.")
    coef = float(mod.params.get(param_name))
    se = float(mod.bse.get(param_name))
    z = coef / se if se != 0 else np.nan
    pval = float(mod.pvalues.get(param_name))
    ci_lower = coef - 1.96 * se
    ci_upper = coef + 1.96 * se
    pct = (math.exp(coef)-1)*100
    ci_pct_lower = (math.exp(ci_lower)-1)*100
    ci_pct_upper = (math.exp(ci_upper)-1)*100
    return {
        'spec': formula,
        'param': param_name,
        'coef': coef,
        'se': se,
        'z': z,
        'pval': pval,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'pct': pct,
        'ci_pct_lower': ci_pct_lower,
        'ci_pct_upper': ci_pct_upper,
        'n_obs': df_local.shape[0],
        'n_pairs': df_local[pair_col].nunique()
    }

# Specifications to run
specs = []

# 1. Baseline DID: log_migrants ~ did + C(pair) + C(year_month) - 1
specs.append(("Baseline DID (pair FE + month FE)", "log_migrants ~ did + C(pair) + C(year_month) - 1"))

# 2. DID with pair-specific linear trends
specs.append(("DID + pair-specific linear trends", "log_migrants ~ did + C(pair) + C(year_month) + C(pair):time_index - 1"))

# 3. DID with pair-specific quadratic trends
df['time_sq'] = df[time_col] ** 2
specs.append(("DID + pair-specific quadratic trends", "log_migrants ~ did + C(pair) + C(year_month) + C(pair):time_index + C(pair):time_sq - 1"))

# 4. Trimmed sample: drop rel_time in [-3,-1]
df['rel_time'] = df[time_col] - treatment_time
df_trim = df[~df['rel_time'].isin([-3,-2,-1])].copy()
specs.append(("Trimmed sample (drop rel_time in [-3,-1])", "log_migrants ~ did + C(pair) + C(year_month) - 1"))

# 5. Optional: trimmed exclude extreme months (example list)
# Replace these months with ones you observed as problematic, e.g. ['2019-01','2020-07', ...]
extreme_months = ['2019-01','2020-07','2020-08','2020-09','2020-10','2020-11','2020-12','2021-01','2021-02','2021-03']
df_noext = df[~df[ym_col].isin(extreme_months)].copy()
specs.append(("Exclude extreme months (COVID & boundary)", "log_migrants ~ did + C(pair) + C(year_month) - 1"))

# Run all specs and collect results. Use correct dataframe for trimmed and noext rows.
results = []
for label, formula in specs:
    if label.startswith("Trimmed sample"):
        info = run_reg(formula, df_trim)
    elif label.startswith("Exclude extreme"):
        info = run_reg(formula, df_noext)
    else:
        info = run_reg(formula, df)
    info['label'] = label
    results.append(info)

# Build table DataFrame
tab = pd.DataFrame(results)[[
    'label','coef','se','z','pval','ci_lower','ci_upper','pct','ci_pct_lower','ci_pct_upper','n_obs','n_pairs'
]]
tab = tab.rename(columns={
    'label':'Specification','coef':'Coef (log)','se':'SE (clustered)','z':'z','pval':'p-value',
    'ci_lower':'95% CI (log) lower','ci_upper':'95% CI (log) upper',
    'pct':'Approx % change','ci_pct_lower':'95% CI (% lower)','ci_pct_upper':'95% CI (% upper)',
    'n_obs':'N obs','n_pairs':'N pairs'
})

# Round nicely for presentation
tab['Coef (log)'] = tab['Coef (log)'].round(6)
tab['SE (clustered)'] = tab['SE (clustered)'].round(6)
tab['z'] = tab['z'].round(2)
tab['p-value'] = tab['p-value'].apply(lambda x: "<0.001" if x<0.001 else round(x,6))
tab['95% CI (log) lower'] = tab['95% CI (log) lower'].round(6)
tab['95% CI (log) upper'] = tab['95% CI (log) upper'].round(6)
tab['Approx % change'] = tab['Approx % change'].apply(lambda x: f"{x:.1f}%")
tab['95% CI (% lower)'] = tab['95% CI (% lower)'].apply(lambda x: f"{x:.1f}%")
tab['95% CI (% upper)'] = tab['95% CI (% upper)'].apply(lambda x: f"{x:.1f}%")

# Compose CI strings
tab['95% CI (log)'] = tab['95% CI (log) lower'].astype(str) + ", " + tab['95% CI (log) upper'].astype(str)
tab['95% CI (% change)'] = "[" + tab['95% CI (% lower)'].astype(str) + ", " + tab['95% CI (% upper)'].astype(str) + "]"

# Select final columns order
final_tab = tab[[
    'Specification','Coef (log)','SE (clustered)','z','p-value','95% CI (log)','Approx % change','95% CI (% change)','N obs','N pairs'
]]

# # Save outputs
# final_tab.to_csv('/mnt/data/tableA1_extended_robustness.csv', index=False)
# # Simple LaTeX (user can adapt to their style)
# with open('/mnt/data/tableA1_extended_robustness.tex','w') as f:
#     f.write(final_tab.to_latex(index=False))

# print("Saved Table A1 to /mnt/data/tableA1_extended_robustness.csv and .tex")
# final_tab
final_tab.T.round(4)

In [ ]:
# Table A2: Full event-study coefficients (±K months)
import numpy as np
import pandas as pd
import re
from patsy import dmatrices
import statsmodels.api as sm
import math

df = BACI_df.copy()
treated_pair = 'CL->US'
pair_col = 'pair'
time_col = 'time_index'
ym_col = 'year_month'
y_col = 'log_migrants'
treatment_time = int(df.loc[(df[pair_col]==treated_pair)&(df['post']==1), time_col].min())

# params
K = 12
min_period, max_period = -K, K

# build safe rel_time names and event dummies (similar to earlier code)
df2 = df.copy()
df2['rel_time'] = df2[time_col] - treatment_time
df2['rel_time_clipped'] = df2['rel_time'].clip(min_period, max_period)

def safe_rel_name(x):
    if x < 0:
        return f'rt_m{abs(int(x))}'
    if x == 0:
        return 'rt_z0'
    return f'rt_p{int(x)}'
df2['rel_label'] = df2['rel_time_clipped'].apply(safe_rel_name)

rel_dummies = pd.get_dummies(df2['rel_label'], prefix='rt')
# drop reference period -1
ref_label = safe_rel_name(-1)
ref_col = f'rt_{ref_label}'
if ref_col in rel_dummies.columns:
    rel_dummies = rel_dummies.drop(columns=[ref_col])

for col in rel_dummies.columns:
    df2[col] = rel_dummies[col].values

is_treated = (df2[pair_col] == treated_pair).astype(int)
evt_cols = []
for col in rel_dummies.columns:
    final = 'evt_' + col.replace('rt_','')
    df2[final] = df2[col] * is_treated
    evt_cols.append(final)

# Fit event-study with pair-specific linear trends
formula = 'log_migrants ~ ' + ' + '.join(evt_cols) + ' + C(pair) + C(year_month) + C(pair):time_index - 1'
y, X = dmatrices(formula, df2, return_type='dataframe')
mod = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': df2[pair_col]})

# Extract coefficients for event dummies and map rel_time
coef = mod.params.filter(like='evt_')
se = mod.bse.filter(like='evt_')

def parse_rel_from_name(name):
    # name like evt_rt_m12 or evt_rt_p3 or evt_rt_z0
    s = name.replace('evt_','')
    if s.startswith('rt_m'):
        return -int(s.split('rt_m')[-1])
    if s.startswith('rt_p'):
        return int(s.split('rt_p')[-1])
    if 'z0' in s:
        return 0
    # fallback
    nums = re.findall(r'-?\d+', s)
    return int(nums[-1]) if nums else np.nan

rows = []
for n in coef.index:
    rel = parse_rel_from_name(n)
    c = float(coef.loc[n])
    s = float(se.loc[n])
    ci_l = c - 1.96*s
    ci_u = c + 1.96*s
    rows.append({'rel_time': rel, 'colname': n, 'coef': c, 'se': s, 'ci_lower': ci_l, 'ci_upper': ci_u})

evt_table = pd.DataFrame(rows).sort_values('rel_time').reset_index(drop=True)

# Add percent change columns if desired
evt_table['pct'] = evt_table['coef'].apply(lambda b: (math.exp(b)-1)*100)
evt_table['pct_ci_lower'] = evt_table['ci_lower'].apply(lambda b: (math.exp(b)-1)*100)
evt_table['pct_ci_upper'] = evt_table['ci_upper'].apply(lambda b: (math.exp(b)-1)*100)

# # Format & save
# evt_table.to_csv('/mnt/data/tableA2_event_study_coeffs.csv', index=False)
# with open('/mnt/data/tableA2_event_study_coeffs.tex','w') as f:
#     f.write(evt_table.to_latex(index=False))

# print("Saved Table A2 to /mnt/data/tableA2_event_study_coeffs.csv and .tex")
evt_table.round(2)


In [ ]:
# FIXED: compute event-study coefficients and save sorted table (t = -K ... +K)
import re
import math
import pandas as pd
from patsy import dmatrices
import statsmodels.api as sm

# --- User settings ---
df = BACI_df.copy()
treated_pair = 'CL->US'
pair_col = 'pair'
time_col = 'time_index'
ym_col = 'year_month'
y_col = 'log_migrants'
# detect treatment_time automatically (same logic as before)
treatment_time = int(df.loc[(df[pair_col] == treated_pair) & (df['post'] == 1), time_col].min())
K = 12   # +/- months to keep
# ----------------------

# 1) make rel_time and safe rel labels
df2 = df.copy()
df2['rel_time'] = df2[time_col] - treatment_time
df2['rel_time_clipped'] = df2['rel_time'].clip(-K, K)

def safe_rel_name(x):
    if x < 0:
        return f'rt_m{abs(int(x))}'
    if x == 0:
        return 'rt_z0'
    return f'rt_p{int(x)}'

df2['rel_label'] = df2['rel_time_clipped'].apply(safe_rel_name)

# 2) one-hot rel_label and drop reference (-1)
rel_dummies = pd.get_dummies(df2['rel_label'], prefix='rt')
ref_label = safe_rel_name(-1)
ref_col = f'rt_{ref_label}'
if ref_col in rel_dummies.columns:
    rel_dummies = rel_dummies.drop(columns=[ref_col])
for col in rel_dummies.columns:
    df2[col] = rel_dummies[col].values

# 3) create treated x rel dummies (only active for treated pair)
df2['is_treated_pair'] = (df2[pair_col] == treated_pair).astype(int)
evt_cols = []
for col in rel_dummies.columns:
    # final name like 'evt_rt_m12' or 'evt_rt_p3' or 'evt_rt_z0' -> consistent and parseable
    final = 'evt_' + col.replace('rt_', '')   # from 'rt_rt_m12' or 'rt_m12' to 'evt_m12' etc.
    df2[final] = df2[col] * df2['is_treated_pair']
    evt_cols.append(final)

# 4) formula with pair-specific linear trends
formula = 'log_migrants ~ ' + ' + '.join(evt_cols) + ' + C(pair) + C(year_month) + C(pair):' + time_col + ' - 1'

# 5) fit model with cluster-robust SE by pair
y, X = dmatrices(formula, df2, return_type='dataframe')
mod = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': df2[pair_col]})

# 6) extract evt coefficients and parse rel_time properly
coef_ser = mod.params.filter(like='evt_')
se_ser = mod.bse.filter(like='evt_')

def parse_rel_from_evt_name(name):
    # name like 'evt_m12', 'evt_p3', 'evt_z0' (we constructed them purposely)
    s = name.replace('evt_', '')
    if s.startswith('m'):
        return -int(s[1:])
    if s.startswith('p'):
        return int(s[1:])
    if s == 'z0':
        return 0
    # fallback: try regex
    m = re.search(r'(-?\d+)', s)
    return int(m.group(1)) if m else None

rows = []
for n in coef_ser.index:
    rel = parse_rel_from_evt_name(n)
    c = float(coef_ser.loc[n])
    s = float(se_ser.loc[n])
    ci_l = c - 1.96*s
    ci_u = c + 1.96*s
    pct = (math.exp(c)-1)*100
    pct_l = (math.exp(ci_l)-1)*100
    pct_u = (math.exp(ci_u)-1)*100
    rows.append({
        'rel_time': rel,
        'colname': n,
        'coef': c,
        'se': s,
        'ci_lower': ci_l,
        'ci_upper': ci_u,
        'pct': pct,
        'pct_ci_lower': pct_l,
        'pct_ci_upper': pct_u
    })

evt_table = pd.DataFrame(rows).sort_values('rel_time').reset_index(drop=True)

# 7) Optional: format numeric columns for presentation
evt_table['coef_round'] = evt_table['coef'].round(3)
evt_table['se_round'] = evt_table['se'].round(3)
evt_table['ci_log'] = evt_table.apply(lambda r: f"[{r['ci_lower']:.3f}, {r['ci_upper']:.3f}]", axis=1)
evt_table['pct_str'] = evt_table['pct'].apply(lambda x: f"{x:.1f}%")
evt_table['pct_ci_str'] = evt_table.apply(lambda r: f"[{r['pct_ci_lower']:.1f}%, {r['pct_ci_upper']:.1f}%]", axis=1)

# # 8) Save sorted CSV for Table A2
# out_csv = '/mnt/data/tableA2_event_study_coeffs_sorted_fixed.csv'
# evt_table.to_csv(out_csv, index=False)
# print("Saved sorted event-study table to:", out_csv)

# 9) Quick display (if in notebook)
display_cols = ['rel_time','colname','coef_round','se_round','ci_log','pct_str','pct_ci_str']
display_df = evt_table[display_cols].rename(columns={
    'rel_time':'Relative month (t)',
    'colname':'Event dummy',
    'coef_round':'Coef (log)',
    'se_round':'SE (clustered)',
    'ci_log':'95% CI (log)',
    'pct_str':'Approx % change',
    'pct_ci_str':'95% CI (% change)'
})

# print(display_df.to_string(index=False))
display_df